In [ ]:
from mpi4py import MPI
import numpy as np
import ufl
from dolfinx.mesh import create_unit_square
from dolfinx.fem import functionspace, Function, Constant, dirichletbc, locate_dofs_geometrical
from dolfinx.fem.petsc import LinearProblem, NonlinearProblem
from basix.ufl import element
from dolfinx.io import VTXWriter

# ----------------------------
# 1. Create mesh
# ----------------------------
n_elem = 32
msh = create_unit_square(MPI.COMM_WORLD, n_elem, n_elem)
tdim = msh.topology.dim
gdim = msh.geometry.dim

# Store original coordinates
x0 = msh.geometry.x.copy()

# ----------------------------
# 2. Scalar function space for solution
# ----------------------------
V = functionspace(msh, ("Lagrange", 1))
u, v = Function(V), ufl.TestFunction(V)
u_n = Function(V)

# Poisson source
f = Constant(msh, 1.0)
dt = Constant(msh, 0.1)
F = (u - u_n)/dt * v * ufl.dx + ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
F += -f * v * ufl.dx

# ----------------------------
# 3. Dirichlet BC on left/bottom edges
# ----------------------------
dofs_D = locate_dofs_geometrical(
    V, lambda x: np.logical_or(np.isclose(x[0], 0), np.isclose(x[1], 0))
)
u_bc = Function(V)
u_bc.interpolate(lambda x: np.zeros(x.shape[1]))
bcs = [dirichletbc(u_bc, dofs_D)]

# ----------------------------
# 4. Vector function space for mesh motion (ALE)
# ----------------------------
V_vec = functionspace(
    msh,
    element("Lagrange", msh.topology.cell_name(), 1, shape=(gdim,))
)
u_mesh = Function(V_vec)

problem = NonlinearProblem(F, u, bcs=bcs, petsc_options_prefix="demo_poisson_",
    petsc_options={"ksp_type": "preonly", "pc_type": "lu", "ksp_error_if_not_converged": True},
)

# ----------------------------
# 5. Time loop with moving mesh and VTU export
# ----------------------------

writer = VTXWriter(MPI.COMM_WORLD, "results/moving_mesh.bp", u, mesh_policy=0)

t = 0.0
while t < 20:
    t += float(dt)
    # --- 5a. Define mesh motion ---
    def mesh_motion(x):
        # x.shape = (gdim, num_points)
        values = np.zeros((gdim, x.shape[1]))
        values[1] = 0.1 * np.sin(np.pi * (x[0] - t/2))  # vertical displacement
        return values

    u_mesh.interpolate(mesh_motion)

    # --- 5b. Update mesh coordinates ---
    array_reshaped = u_mesh.x.array.reshape(msh.geometry.x.shape[0], gdim)
    zero_col = np.zeros((array_reshaped.shape[0], 1))
    array_appended = np.hstack([array_reshaped, zero_col])

    msh.geometry.x[:] = x0 + array_appended

    # --- 5d. Solve Poisson problem ---

    problem.solve()

    u_n.x.array[:] = u.x.array[:]

    # --- 5e. Write solution at current timestep ---
    writer.write(t)
    # print(t)

In [ ]:
import festim as F
import ufl
import matplotlib.pyplot as plt
import numpy as np
import dolfinx
from mpi4py import MPI

# Create and mark the mesh
nx = ny = 10
fenics_mesh = dolfinx.mesh.create_unit_square(MPI.COMM_WORLD, nx, ny)

def my_custom_model(model):
    print("hola!")
    model.temperature = 300
    
    return model
     

In [ ]:
my_model = F.HydrogenTransportProblem()
my_model.mesh = fenics_mesh

custom_model = my_custom_model(model=my_model)
print(custom_model.temperature)
print(custom_model.mesh)

In [ ]:
import festim as F
import ufl
import matplotlib.pyplot as plt
import numpy as np
import dolfinx
from mpi4py import MPI

# Create and mark the mesh
nx = ny = 10
fenics_mesh = dolfinx.mesh.create_unit_square(MPI.COMM_WORLD, nx, ny)

# Create the FESTIM model
my_model = F.HydrogenTransportProblem()

H = F.Species("H")
my_model.species = [H]
my_model.mesh = F.Mesh(fenics_mesh)

D = 2.0
material = F.Material(D_0=D, E_D=0)

volume = F.VolumeSubdomain(id=1, material=material)
boundary = F.SurfaceSubdomain(id=1)

my_model.subdomains = [boundary, volume]


my_model.boundary_conditions = [
    F.FixedConcentrationBC(subdomain=boundary, value=0, species=H),
]


my_model.temperature = 500

my_model.settings = F.Settings(
    atol=1e-10,
    rtol=1e-10,
    transient=False,
)

my_model.initialise()
my_model.run()

In [32]:
import festim as F
import dolfinx
import numpy as np
from mpi4py import MPI
from dolfinx.fem import functionspace, Function
from basix.ufl import element

nx = ny = 32
fenics_mesh = dolfinx.mesh.create_unit_square(MPI.COMM_WORLD, nx, ny)
gdim = fenics_mesh.geometry.dim

# Reference (undeformed) coordinates – we always displace FROM here
x0 = fenics_mesh.geometry.x.copy()

# ALE displacement function space (vector Lagrange on the same mesh)
V_ale = functionspace(
    fenics_mesh,
    element("Lagrange", fenics_mesh.topology.cell_name(), 1, shape=(gdim,))
)
u_mesh = Function(V_ale)

# ─────────────────────────────────────────────
# 2. Define the mesh-motion function
#    Signature matches dolfinx interpolation: x.shape = (gdim, n_points)
# ─────────────────────────────────────────────
def mesh_motion(x, t):
    values = np.zeros((gdim, x.shape[1]))
    values[1] = 0.1 * np.sin(np.pi * (x[0] - t / 8))   # gentle vertical ripple
    return values

# ─────────────────────────────────────────────
# 3. Helper: update mesh geometry to time t
# ─────────────────────────────────────────────
def update_mesh(t):
    u_mesh.interpolate(lambda x: mesh_motion(x, t))

    disp = u_mesh.x.array.reshape(-1, gdim)  # shape (n_nodes, 2)

    # dolfinx geometry.x is always (n_nodes, 3); pad the 2-D displacement
    disp3 = np.hstack([disp, np.zeros((disp.shape[0], 1))])

    fenics_mesh.geometry.x[:] = x0 + disp3

# ─────────────────────────────────────────────
# 4. Build and initialise the FESTIM model
#    (do NOT call model.run() – we drive the loop ourselves)
# ─────────────────────────────────────────────
H = F.Species("H")
material = F.Material(D_0=1e-2, E_D=0)

volume   = F.VolumeSubdomain(id=1, material=material)
top_surface = F.SurfaceSubdomain(id=1, locator = lambda x: np.isclose(x[0], 1.0))
bottom_surface = F.SurfaceSubdomain(id=2, locator = lambda x: np.isclose(x[0], 0.0))

model = F.HydrogenTransportProblem()
model.mesh     = F.Mesh(fenics_mesh)
model.species  = [H]
model.subdomains = [top_surface, bottom_surface, volume]
model.boundary_conditions = [
    F.FixedConcentrationBC(subdomain=left_surface, value=1, species=H),
    F.FixedConcentrationBC(subdomain=right_surface, value=0, species=H)
]
model.temperature = 500

dt_value = 0.5
final_time = 50

model.settings = F.Settings(
    atol=1e-10,
    rtol=1e-10,
    transient=True,
    final_time=final_time,
    stepsize=F.Stepsize(dt_value),
)

model.exports = [F.VTXSpeciesExport(filename="results/moving_mesh/simple.bp", field=H)]
model.initialise()   # builds FEM spaces, forms, solver – but does NOT time-step yet

# ─────────────────────────────────────────────
# 5. Custom time loop with ALE mesh motion
# ─────────────────────────────────────────────
t = 0.0
model.show_progress_bar = False  # <-- fixes the AttributeError

while model.t.value < model.settings.final_time:

    # (a) Move mesh to the NEXT time (t + dt) before solving
    next_t = model.t.value + float(model._dt)
    update_mesh(next_t)

    # (b) iterate() internally does: t += dt, update BCs, solve, post-process, u_n <- u
    model.iterate()

    print(f"t = {model.t.value:.3f}  |  max(H) = {H.post_processing_solution.x.array.max():.4e}")

t = 0.500  |  max(H) = 1.0000e+00
t = 1.000  |  max(H) = 1.0000e+00
t = 1.500  |  max(H) = 1.0000e+00
t = 2.000  |  max(H) = 1.0000e+00
t = 2.500  |  max(H) = 1.0000e+00
t = 3.000  |  max(H) = 1.0000e+00
t = 3.500  |  max(H) = 1.0000e+00
t = 4.000  |  max(H) = 1.0000e+00
t = 4.500  |  max(H) = 1.0000e+00
t = 5.000  |  max(H) = 1.0000e+00
t = 5.500  |  max(H) = 1.0000e+00
t = 6.000  |  max(H) = 1.0000e+00
t = 6.500  |  max(H) = 1.0000e+00
t = 7.000  |  max(H) = 1.0000e+00
t = 7.500  |  max(H) = 1.0000e+00
t = 8.000  |  max(H) = 1.0000e+00
t = 8.500  |  max(H) = 1.0000e+00
t = 9.000  |  max(H) = 1.0000e+00
t = 9.500  |  max(H) = 1.0000e+00
t = 10.000  |  max(H) = 1.0000e+00
t = 10.500  |  max(H) = 1.0000e+00
t = 11.000  |  max(H) = 1.0000e+00
t = 11.500  |  max(H) = 1.0000e+00
t = 12.000  |  max(H) = 1.0000e+00
t = 12.500  |  max(H) = 1.0000e+00
t = 13.000  |  max(H) = 1.0000e+00
t = 13.500  |  max(H) = 1.0000e+00
t = 14.000  |  max(H) = 1.0000e+00
t = 14.500  |  max(H) = 1.0000e+00
t = 